# Task 1: Exploratory Data Analysis — ACIS Insurance Data

## Business Context
AlphaCare Insurance Solutions (ACIS) wants to optimize its car insurance marketing and pricing strategy in South Africa. This EDA explores historical policy, client, vehicle, and claim data (Feb 2014 – Aug 2015) to uncover risk patterns, profitability drivers, and segmentation opportunities.

**Key Metrics:**
- **Loss Ratio** = TotalClaims / TotalPremium (lower is more profitable)
- **Claim Frequency** = proportion of policies with at least one claim
- **Claim Severity** = average claim amount given a claim occurred

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

print('Libraries imported successfully')

## 1. Load Data

In [ ]:
import os

paths = [
    '../data/MachineLearningRating_v3.txt',
    r'c:\KAIM\MachineLearningRating_v3.txt',
    '../data/insurance_data.txt',
]

df = None
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path, sep='|', low_memory=False)
        print(f'Loaded from: {path}')
        break

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Data Summarization & Quality Assessment

In [ ]:
# Fix data types
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'], errors='coerce')
df['TotalPremium'] = pd.to_numeric(df['TotalPremium'], errors='coerce')
df['TotalClaims'] = pd.to_numeric(df['TotalClaims'], errors='coerce')
df['CustomValueEstimate'] = pd.to_numeric(df['CustomValueEstimate'], errors='coerce')
df['CalculatedPremiumPerTerm'] = pd.to_numeric(df['CalculatedPremiumPerTerm'], errors='coerce')
df['SumInsured'] = pd.to_numeric(df['SumInsured'], errors='coerce')

print('=== Data Types ===')
print(df.dtypes)
print(f'\nDate range: {df["TransactionMonth"].min()} to {df["TransactionMonth"].max()}')

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('=== Missing Values ===')
print(missing_df)
print(f'\nHandling Strategy:')
print('- Columns with >50% missing: drop for modeling, keep for EDA')
print('- Numerical columns: impute with median')
print('- Categorical columns: impute with mode or "Unknown"')

In [ ]:
# Descriptive statistics for key numerical features
num_cols = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate',
            'SumInsured', 'CalculatedPremiumPerTerm']
print('=== Descriptive Statistics ===')
print(df[num_cols].describe().round(2))

## 3. Loss Ratio Analysis
**Loss Ratio = TotalClaims / TotalPremium**
A loss ratio > 1 means the insurer is paying out more in claims than it collects in premiums — unprofitable.

In [ ]:
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium'].replace(0, np.nan)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

overall_lr = df['TotalClaims'].sum() / df['TotalPremium'].sum()
print(f'Overall Loss Ratio: {overall_lr:.4f} ({overall_lr*100:.2f}%)')
print(f'Overall Claim Frequency: {df["HasClaim"].mean()*100:.2f}%')
print(f'\nLoss Ratio by Province:')
province_lr = df.groupby('Province').apply(
    lambda x: x['TotalClaims'].sum() / x['TotalPremium'].sum()
).round(4).sort_values(ascending=False)
print(province_lr)

print(f'\nLoss Ratio by VehicleType:')
vehicle_lr = df.groupby('VehicleType').apply(
    lambda x: x['TotalClaims'].sum() / x['TotalPremium'].sum()
).round(4).sort_values(ascending=False)
print(vehicle_lr)

print(f'\nLoss Ratio by Gender:')
gender_lr = df.groupby('Gender').apply(
    lambda x: x['TotalClaims'].sum() / x['TotalPremium'].sum()
).round(4)
print(gender_lr)

## 4. Visualization 1 — Loss Ratio by Province & Vehicle Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Loss ratio by province
province_lr_plot = province_lr.reset_index()
province_lr_plot.columns = ['Province', 'LossRatio']
colors = ['#e74c3c' if x > 1 else '#2ecc71' for x in province_lr_plot['LossRatio']]
axes[0].barh(province_lr_plot['Province'], province_lr_plot['LossRatio'], color=colors)
axes[0].axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Break-even (1.0)')
axes[0].axvline(overall_lr, color='blue', linestyle=':', linewidth=1.5,
                label=f'Portfolio avg ({overall_lr:.2f})')
axes[0].set_title('Loss Ratio by Province', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Loss Ratio')
axes[0].legend()

# Loss ratio by vehicle type
vt_lr = vehicle_lr.head(10).reset_index()
vt_lr.columns = ['VehicleType', 'LossRatio']
colors2 = ['#e74c3c' if x > 1 else '#3498db' for x in vt_lr['LossRatio']]
axes[1].barh(vt_lr['VehicleType'], vt_lr['LossRatio'], color=colors2)
axes[1].axvline(1.0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title('Loss Ratio by Vehicle Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Loss Ratio')

plt.suptitle('Loss Ratio Analysis — Red = Unprofitable (>1.0)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_ratio_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved!')

## 5. Univariate Analysis — Key Financial Variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# TotalPremium distribution
prem_clip = df['TotalPremium'].clip(0, df['TotalPremium'].quantile(0.99))
axes[0,0].hist(prem_clip, bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('TotalPremium Distribution', fontweight='bold')
axes[0,0].set_xlabel('Premium (ZAR)')
axes[0,0].set_ylabel('Frequency')

# TotalClaims distribution
claims_nonzero = df[df['TotalClaims'] > 0]['TotalClaims']
claims_clip = claims_nonzero.clip(0, claims_nonzero.quantile(0.99))
axes[0,1].hist(claims_clip, bins=50, color='coral', edgecolor='white')
axes[0,1].set_title('TotalClaims Distribution (Non-zero)', fontweight='bold')
axes[0,1].set_xlabel('Claims (ZAR)')

# CustomValueEstimate
cve = df['CustomValueEstimate'].dropna()
cve_clip = cve.clip(0, cve.quantile(0.99))
axes[0,2].hist(cve_clip, bins=50, color='purple', edgecolor='white', alpha=0.7)
axes[0,2].set_title('CustomValueEstimate Distribution', fontweight='bold')
axes[0,2].set_xlabel('Vehicle Value (ZAR)')

# Province bar chart
prov_counts = df['Province'].value_counts().head(10)
axes[1,0].barh(prov_counts.index[::-1], prov_counts.values[::-1],
               color=sns.color_palette('viridis', len(prov_counts)))
axes[1,0].set_title('Policies by Province', fontweight='bold')
axes[1,0].set_xlabel('Count')

# Vehicle type bar chart
vt_counts = df['VehicleType'].value_counts().head(8)
axes[1,1].barh(vt_counts.index[::-1], vt_counts.values[::-1],
               color=sns.color_palette('plasma', len(vt_counts)))
axes[1,1].set_title('Policies by Vehicle Type', fontweight='bold')
axes[1,1].set_xlabel('Count')

# Gender distribution
gender_counts = df['Gender'].value_counts()
axes[1,2].bar(gender_counts.index, gender_counts.values,
              color=['#3498db', '#e74c3c', '#95a5a6'])
axes[1,2].set_title('Policies by Gender', fontweight='bold')
axes[1,2].set_xlabel('Gender')
axes[1,2].set_ylabel('Count')

plt.suptitle('Univariate Analysis — Key Variables', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('univariate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved!')

## 6. Outlier Detection — Box Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, col, color in zip(axes,
    ['TotalPremium', 'TotalClaims', 'CustomValueEstimate'],
    ['steelblue', 'coral', 'purple']):
    data = df[col].dropna()
    data = data[data > 0]
    q99 = data.quantile(0.99)
    ax.boxplot(data.clip(0, q99), patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6))
    ax.set_title(f'{col}\n(clipped at 99th percentile)', fontweight='bold')
    ax.set_ylabel('ZAR')
    outliers = data[data > q99]
    ax.text(1.1, q99, f'{len(outliers):,} outliers\n(>{q99:,.0f})',
            fontsize=9, color='red')

plt.suptitle('Outlier Detection — Key Financial Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved!')

## 7. Bivariate Analysis — TotalPremium vs TotalClaims

In [ ]:
# Correlation matrix
corr_cols = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate',
             'SumInsured', 'CalculatedPremiumPerTerm']
corr = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation heatmap
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[0], center=0, square=True)
axes[0].set_title('Correlation Matrix — Financial Variables', fontweight='bold')

# Scatter plot - Premium vs Claims by Province
sample = df[df['TotalClaims'] > 0].sample(min(5000, len(df)), random_state=42)
provinces = sample['Province'].unique()[:6]
colors_map = dict(zip(provinces, sns.color_palette('tab10', len(provinces))))
for prov in provinces:
    mask = sample['Province'] == prov
    axes[1].scatter(
        sample[mask]['TotalPremium'].clip(0, 5000),
        sample[mask]['TotalClaims'].clip(0, 50000),
        alpha=0.3, s=10, label=prov, color=colors_map[prov]
    )
axes[1].set_title('TotalPremium vs TotalClaims by Province', fontweight='bold')
axes[1].set_xlabel('TotalPremium (ZAR)')
axes[1].set_ylabel('TotalClaims (ZAR)')
axes[1].legend(fontsize=8)

plt.suptitle('Bivariate Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bivariate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved!')

## 8. Visualization 2 — Temporal Trends

In [ ]:
df['YearMonth'] = df['TransactionMonth'].dt.to_period('M')
monthly = df.groupby('YearMonth').agg(
    TotalPremium=('TotalPremium', 'sum'),
    TotalClaims=('TotalClaims', 'sum'),
    ClaimFreq=('HasClaim', 'mean'),
    PolicyCount=('PolicyID', 'count')
).reset_index()
monthly['LossRatio'] = monthly['TotalClaims'] / monthly['TotalPremium']
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].plot(monthly['YearMonth_str'], monthly['TotalPremium'],
               color='steelblue', marker='o', markersize=3)
axes[0,0].set_title('Monthly Total Premium', fontweight='bold')
axes[0,0].set_ylabel('ZAR')
axes[0,0].tick_params(axis='x', rotation=45)

axes[0,1].plot(monthly['YearMonth_str'], monthly['TotalClaims'],
               color='coral', marker='o', markersize=3)
axes[0,1].set_title('Monthly Total Claims', fontweight='bold')
axes[0,1].set_ylabel('ZAR')
axes[0,1].tick_params(axis='x', rotation=45)

axes[1,0].plot(monthly['YearMonth_str'], monthly['LossRatio'],
               color='purple', marker='o', markersize=3)
axes[1,0].axhline(1.0, color='red', linestyle='--', label='Break-even')
axes[1,0].set_title('Monthly Loss Ratio', fontweight='bold')
axes[1,0].set_ylabel('Loss Ratio')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].legend()

axes[1,1].plot(monthly['YearMonth_str'], monthly['ClaimFreq']*100,
               color='green', marker='o', markersize=3)
axes[1,1].set_title('Monthly Claim Frequency (%)', fontweight='bold')
axes[1,1].set_ylabel('Claim Frequency (%)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Temporal Trends — Feb 2014 to Aug 2015', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('temporal_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 5 saved!')

## 9. Visualization 3 — Vehicle Make Analysis

In [ ]:
make_stats = df.groupby('make').agg(
    AvgClaims=('TotalClaims', 'mean'),
    AvgPremium=('TotalPremium', 'mean'),
    PolicyCount=('PolicyID', 'count'),
    ClaimFreq=('HasClaim', 'mean')
).reset_index()
make_stats['LossRatio'] = make_stats['AvgClaims'] / make_stats['AvgPremium'].replace(0, np.nan)
make_stats = make_stats[make_stats['PolicyCount'] >= 100]

top_claims = make_stats.nlargest(10, 'AvgClaims')
low_claims = make_stats.nsmallest(10, 'AvgClaims')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top_claims['make'], top_claims['AvgClaims'],
             color='#e74c3c')
axes[0].set_title('Top 10 Makes — Highest Avg Claims', fontweight='bold')
axes[0].set_xlabel('Average Claims (ZAR)')

axes[1].barh(low_claims['make'], low_claims['AvgClaims'],
             color='#2ecc71')
axes[1].set_title('Top 10 Makes — Lowest Avg Claims', fontweight='bold')
axes[1].set_xlabel('Average Claims (ZAR)')

plt.suptitle('Vehicle Make Risk Analysis (min 100 policies)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('vehicle_make_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 6 saved!')

## 10. Geographic Trends

In [ ]:
geo = df.groupby('Province').agg(
    AvgPremium=('TotalPremium', 'mean'),
    AvgClaims=('TotalClaims', 'mean'),
    ClaimFreq=('HasClaim', 'mean'),
    PolicyCount=('PolicyID', 'count')
).round(2)
geo['LossRatio'] = (geo['AvgClaims'] / geo['AvgPremium']).round(4)
geo = geo.sort_values('LossRatio', ascending=False)

print('=== Geographic Risk Summary ===')
print(geo.to_string())

# Top make per province
print('\n=== Most Common Vehicle Make by Province ===')
top_make = df.groupby('Province')['make'].agg(lambda x: x.value_counts().index[0])
print(top_make)

## 11. Key Insights & Summary

### Findings:
1. **Overall Loss Ratio** indicates whether the portfolio is profitable overall
2. **Provincial Risk Variation** — some provinces show significantly higher loss ratios, suggesting need for regional pricing adjustments
3. **Outliers** — TotalClaims has extreme outliers that could skew models; capping at 99th percentile recommended
4. **Temporal Trends** — claim frequency and severity vary over the 18-month period, possibly tied to seasonal factors
5. **Vehicle Make Risk** — luxury and heavy vehicles tend to have higher average claim amounts
6. **Gender** — difference in loss ratio between genders may warrant adjusted pricing

### Next Steps (Task 2):
- Set up DVC for data version control
- Track raw and cleaned versions of the dataset
- Establish reproducible data pipeline

In [ ]:
# Save cleaned version for next tasks
cols_to_keep = [
    'PolicyID', 'TransactionMonth', 'Province', 'PostalCode',
    'Gender', 'MaritalStatus', 'VehicleType', 'make', 'Model',
    'RegistrationYear', 'CustomValueEstimate', 'SumInsured',
    'CalculatedPremiumPerTerm', 'TotalPremium', 'TotalClaims',
    'CoverType', 'CoverCategory', 'LossRatio', 'Margin', 'HasClaim'
]
df_clean = df[cols_to_keep].copy()
df_clean.to_csv('../data/insurance_cleaned.csv', index=False)
print(f'Cleaned data saved: {df_clean.shape}')
print(f'Missing values remaining: {df_clean.isnull().sum().sum()}')